# OBJECTIVE

# Refactor the delivery-time prediction code built in P02/P03 from scattered notebook cells into a proper Python package (delivery/) with separate modules — data.py, features.py, model.py, and validate.py — tied together with an __init__.py. Import and use this package like a library to train and save a model, then run the full pipeline from the command line using standalone scripts (train.py and predict.py), without relying on a notebook.

# Tasks:

# T1: Add a function average_speed_kmph(distance_km, delivery_min) to the package.

# T2: Add a delivery/validate.py module with a function that rejects an impossible/unrealistic order.

# T3: Write a second command-line script, predict.py, that loads the saved model and predicts delivery time for a new order.

# Create Folders

In [1]:
#`delivery/` will hold our package files, `data/` will hold the CSV.

In [2]:
import os
os.makedirs("delivery", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Folders ready")

Folders ready


# Step 1: data.py --- loads the data

In [3]:
%%writefile delivery/data.py
import pandas as pd
import numpy as np
import os

def load_data():
    path = "data/delivery_times.csv"

    if not os.path.exists(path):
        np.random.seed(42)
        n = 600
        distance_km = np.random.uniform(0.5, 12, n)
        prep_time_min = np.random.uniform(5, 30, n)
        traffic_level = np.random.randint(1, 4, n)
        rain = np.random.randint(0, 2, n)
        noise = np.random.normal(0, 2, n)

        delivery_min = 6 + 3 * distance_km + 0.6 * prep_time_min + 4 * traffic_level + 5 * rain + noise

        df = pd.DataFrame({
            "distance_km": distance_km,
            "prep_time_min": prep_time_min,
            "traffic_level": traffic_level,
            "rain": rain,
            "delivery_min": delivery_min
        })
        df.to_csv(path, index=False)

    return pd.read_csv(path)

Overwriting delivery/data.py


# Step 2: features.py --- splits X and y (also has T1)

# Splits the data into inputs (X) and the target to predict (y).
# average_speed_kmph is Task T1.

In [4]:
%%writefile delivery/features.py
def get_features_and_target(df):
    X = df[["distance_km", "prep_time_min", "traffic_level", "rain"]]
    y = df["delivery_min"]
    return X, y

# T1
def average_speed_kmph(distance_km, delivery_min):
    hours = delivery_min / 60
    return distance_km / hours

Overwriting delivery/features.py


# Step 3: model.py --- trains, checks, and saves the model

# Trains a LinearRegression model, prints its test error (MAE), and saves it to a file so predict.py can reuse it later.

In [5]:
%%writefile delivery/model.py
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

def train_and_save_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print("Test MAE:", round(mae, 2))

    joblib.dump(model, "delivery_model.joblib")
    return model

Overwriting delivery/model.py


# Step 4: validate.py --- T2, rejects a bad order
# Checks that an order's values are realistic before we predict on it.

In [6]:
%%writefile delivery/validate.py
# T2
def is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    if distance_km <= 0:
        return False
    if prep_time_min <= 0:
        return False
    if traffic_level not in [1, 2, 3]:
        return False
    if rain not in [0, 1]:
        return False
    return True

Overwriting delivery/validate.py


# Step 5: __init__.py --- makes delivery a package

# Built last so it can import functions from all the other files. This is what makes `from delivery import ...` work.

In [7]:
%%writefile delivery/__init__.py
from .data import load_data
from .features import get_features_and_target, average_speed_kmph
from .model import train_and_save_model
from .validate import is_valid_order

Overwriting delivery/__init__.py


# Step 6: Import and use our own package
# Proves the package works --- import it like a library and train the model.

In [8]:
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

Test MAE: 1.4


# Step 7: train.py  script

In [9]:
%%writefile train.py
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Overwriting train.py


# Step 8: Run train.py like a real program
# Runs the script as a terminal command, no notebook involved.

In [10]:
!python train.py

Test MAE: 1.4
Training done


# T3: predict.py --- second script
# Loads the saved model and predicts delivery time for one new order.

In [11]:
%%writefile predict.py
import sys
import joblib
import pandas as pd
from delivery import is_valid_order

distance_km = float(sys.argv[1])
prep_time_min = float(sys.argv[2])
traffic_level = int(sys.argv[3])
rain = int(sys.argv[4])

if not is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    print("Invalid order")
else:
    model = joblib.load("delivery_model.joblib")
    order = pd.DataFrame([[distance_km, prep_time_min, traffic_level, rain]],
                          columns=["distance_km", "prep_time_min", "traffic_level", "rain"])
    prediction = model.predict(order)[0]
    print("Predicted delivery time:", round(prediction, 1), "minutes")

Overwriting predict.py


# Test predict.py --- a normal order

In [12]:
!python predict.py 5.0 15 2 0

Predicted delivery time: 37.9 minutes


# Test predict.py --- a bad order (T2 check)

In [13]:
!python predict.py -3 15 2 0

Invalid order


# Test T1: average_speed_kmph

In [14]:
from delivery import average_speed_kmph

speed = average_speed_kmph(distance_km=5.0, delivery_min=40.4)
print("Average speed (km/h):", round(speed, 2))

Average speed (km/h): 7.43


# Summary


- data.py --- loads the delivery data
- features.py --- splits data into X and y
- model.py --- trains, checks, and saves the model
- validate.py --- rejects a bad order
- __init__.py --- makes the folder a package

# To Do: Package the classification workflow

Using the same pattern from this lab (data.py -> features.py -> model.py ->
validate.py -> __init__.py), turn the classification code from Lab 3
(Breast Cancer dataset) into its own package. This time, go a step further
than just training one fixed model.

T1 --- model.py should not train just one model. Use cross-validation to
compare LogisticRegression, DecisionTreeClassifier, and
RandomForestClassifier, and automatically save whichever one scores best
(instead of hardcoding the winner yourself).

T2 --- validate.py should check that at least 3 of the input measurements
fall within a realistic range (e.g. radius_mean and area_mean can't be
negative or absurdly large) --- not just "is this a number".

T3 --- predict.py should print both the prediction (malignant/benign) AND
the model's confidence for that prediction (use predict_proba).

T4 --- add a metrics.py module with one function that prints a
confusion matrix and classification report for the saved model, so anyone
can check its performance without retraining.

In [15]:
import os

os.makedirs("breast_cancer", exist_ok=True)

print("breast_cancer package created successfully!")

breast_cancer package created successfully!


In [16]:
%%writefile breast_cancer/data.py

from sklearn.datasets import load_breast_cancer
import pandas as pd


def load_data():

    data = load_breast_cancer()

    X = pd.DataFrame(
        data.data,
        columns=data.feature_names
    )

    y = pd.Series(
        data.target,
        name="target"
    )

    return X, y

Overwriting breast_cancer/data.py


In [17]:
%%writefile breast_cancer/features.py

def get_features_and_target(X, y):
    return X, y

Overwriting breast_cancer/features.py


In [18]:
%%writefile breast_cancer/validate.py


def validate_input(mean_radius, mean_texture, mean_area):

    # Check mean radius
    if mean_radius <= 0 or mean_radius > 50:
        return False

    # Check mean texture
    if mean_texture <= 0 or mean_texture > 100:
        return False

    # Check mean area
    if mean_area <= 0 or mean_area > 5000:
        return False

    return True

Overwriting breast_cancer/validate.py


In [19]:
%%writefile breast_cancer/model.py

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import joblib


def train_and_save_model(X, y):

    models = {

        "Logistic Regression":
            LogisticRegression(
                max_iter=10000
            ),

        "Decision Tree":
            DecisionTreeClassifier(
                max_depth=4,
                random_state=42
            ),

        "Random Forest":
            RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )
    }

    best_model = None
    best_score = 0
    best_name = ""

    # Compare all models
    for name, model in models.items():

        scores = cross_val_score(
            model,
            X,
            y,
            cv=5,
            scoring="accuracy"
        )

        mean_score = scores.mean()

        print(
            f"{name}: "
            f"CV Accuracy = {mean_score:.4f}"
        )

        # Automatically select best model
        if mean_score > best_score:

            best_score = mean_score
            best_model = model
            best_name = name

    # Train best model on complete dataset
    best_model.fit(X, y)

    # Save best model
    joblib.dump(
        best_model,
        "breast_cancer_model.joblib"
    )

    print("\nBest Model:", best_name)
    print(
        "Best CV Accuracy:",
        round(best_score, 4)
    )

    print(
        "Model saved as "
        "breast_cancer_model.joblib"
    )

    return best_model

Overwriting breast_cancer/model.py


In [20]:
%%writefile breast_cancer/__init__.py

from .data import load_data
from .features import get_features_and_target
from .model import train_and_save_model
from .validate import validate_input

Overwriting breast_cancer/__init__.py


In [21]:
import sys

# Remove all cached breast_cancer modules
for module in list(sys.modules.keys()):
    if module == "breast_cancer" or module.startswith("breast_cancer."):
        del sys.modules[module]

print("Old breast_cancer modules cleared!")

Old breast_cancer modules cleared!


In [22]:
from breast_cancer import (
    load_data,
    get_features_and_target,
    train_and_save_model,
    validate_input
)

print("Package imported successfully!")

Package imported successfully!


In [23]:
X, y = load_data()

X, y = get_features_and_target(X, y)

model = train_and_save_model(X, y)

Logistic Regression: CV Accuracy = 0.9508
Decision Tree: CV Accuracy = 0.9209
Random Forest: CV Accuracy = 0.9561

Best Model: Random Forest
Best CV Accuracy: 0.9561
Model saved as breast_cancer_model.joblib


In [24]:
result = validate_input(
    mean_radius=14.0,
    mean_texture=20.0,
    mean_area=600.0
)

print("Validation result:", result)

Validation result: True


In [25]:
result = validate_input(
    mean_radius=-5.0,
    mean_texture=20.0,
    mean_area=600.0
)

print("Validation result:", result)

Validation result: False


In [26]:
%%writefile breast_cancer/predict.py

import sys
import joblib
import pandas as pd

from sklearn.datasets import load_breast_cancer
from .validate import validate_input


def predict_sample(values):

    data = load_breast_cancer()

    # Create a DataFrame with the correct 30 feature names
    sample = pd.DataFrame(
        [values],
        columns=data.feature_names
    )

    # Get the 3 measurements we validate
    mean_radius = sample["mean radius"].iloc[0]
    mean_texture = sample["mean texture"].iloc[0]
    mean_area = sample["mean area"].iloc[0]

    # Validate input
    if not validate_input(
        mean_radius,
        mean_texture,
        mean_area
    ):
        print("Invalid input measurements.")
        return

    # Load the saved model
    model = joblib.load(
        "breast_cancer_model.joblib"
    )

    # Make prediction
    prediction = model.predict(sample)[0]

    # Get prediction probabilities
    probabilities = model.predict_proba(sample)[0]

    # Confidence of predicted class
    confidence = probabilities[prediction] * 100

    # Convert 0/1 to malignant/benign
    result = data.target_names[prediction]

    print("Prediction:", result)
    print(f"Confidence: {confidence:.2f}%")


if __name__ == "__main__":

    values = [
        float(value)
        for value in sys.argv[1:]
    ]

    if len(values) != 30:
        print(
            "Error: Please provide exactly "
            "30 feature values."
        )
    else:
        predict_sample(values)

Overwriting breast_cancer/predict.py


In [27]:
sample = X.iloc[0].values

print("Number of features:", len(sample))
print("Sample prepared successfully!")

Number of features: 30
Sample prepared successfully!


In [28]:
import subprocess

values = " ".join(map(str, sample))

result = subprocess.run(
    ["python", "-m", "breast_cancer.predict"] + values.split(),
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

Prediction: malignant
Confidence: 96.00%



In [29]:
%%writefile breast_cancer/metrics.py

import joblib
import pandas as pd

from sklearn.datasets import load_breast_cancer

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)


def evaluate_saved_model():

    # Load the dataset
    data = load_breast_cancer()

    X = pd.DataFrame(
        data.data,
        columns=data.feature_names
    )

    y = data.target

    # Load the already saved model
    model = joblib.load(
        "breast_cancer_model.joblib"
    )

    # Make predictions
    predictions = model.predict(X)

    # Confusion Matrix
    cm = confusion_matrix(
        y,
        predictions
    )

    print("Confusion Matrix:")
    print(cm)

    # Classification Report
    print("\nClassification Report:")

    print(
        classification_report(
            y,
            predictions,
            target_names=data.target_names
        )
    )

Overwriting breast_cancer/metrics.py


In [30]:
from breast_cancer.metrics import evaluate_saved_model

evaluate_saved_model()

Confusion Matrix:
[[212   0]
 [  0 357]]

Classification Report:
              precision    recall  f1-score   support

   malignant       1.00      1.00      1.00       212
      benign       1.00      1.00      1.00       357

    accuracy                           1.00       569
   macro avg       1.00      1.00      1.00       569
weighted avg       1.00      1.00      1.00       569



In [31]:
import os
import joblib

checks = []

# 1. Package exists
checks.append(("breast_cancer package", os.path.isdir("breast_cancer")))

# 2. __init__.py exists
checks.append(("__init__.py", os.path.isfile("breast_cancer/__init__.py")))

# 3. data.py exists
checks.append(("data.py", os.path.isfile("breast_cancer/data.py")))

# 4. features.py exists
checks.append(("features.py", os.path.isfile("breast_cancer/features.py")))

# 5. model.py exists
checks.append(("model.py", os.path.isfile("breast_cancer/model.py")))

# 6. validate.py exists
checks.append(("validate.py", os.path.isfile("breast_cancer/validate.py")))

# 7. predict.py exists
checks.append(("predict.py", os.path.isfile("breast_cancer/predict.py")))

# 8. metrics.py exists
checks.append(("metrics.py", os.path.isfile("breast_cancer/metrics.py")))

# 9. Saved model exists and can be loaded
try:
    model = joblib.load("breast_cancer_model.joblib")
    model_ok = model is not None
except Exception:
    model_ok = False

checks.append(("Saved model", model_ok))


passed = sum(result for _, result in checks)
total = len(checks)

print("=" * 45)
print("PRACTICAL 04 SELF-CHECK")
print("=" * 45)

for name, result in checks:
    print(f"{'✓' if result else '✗'} {name}")

print("-" * 45)
print(f"{passed} of {total} checks passed")

PRACTICAL 04 SELF-CHECK
✓ breast_cancer package
✓ __init__.py
✓ data.py
✓ features.py
✓ model.py
✓ validate.py
✓ predict.py
✓ metrics.py
✓ Saved model
---------------------------------------------
9 of 9 checks passed
